# Middleware — 中间件与回调

对标文档：LangChain Core Components > Callbacks

In [1]:
from langchain_learning.llm import get_llm
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import time

llm = get_llm()

**术语：**
- **Middleware（中间件）** = 在 LLM 调用前后插入的自定义逻辑
- **Callback（回调）** = 在特定事件发生时触发的函数，是实现中间件的主要方式
- **事件类型**：开始(on_start)、结束(on_end)、报错(on_error)、新 token(on_llm_new_token)

### 1. 日志回调——记录每次调用

In [2]:
class LoggingHandler(BaseCallbackHandler):
    def on_llm_start(self, serialized, prompts, **kwargs):
        print(f"[LLM 开始] 输入: {prompts[0][:50]}...")

    def on_llm_end(self, response, **kwargs):
        print(f"[LLM 结束] 输出: {response.generations[0][0].text[:50]}...")

    def on_llm_error(self, error, **kwargs):
        print(f"[LLM 错误] {error}")

logger = LoggingHandler()
result = llm.invoke("用一句话介绍 Python", config={"callbacks": [logger]})
print("回复:", result.content)

[LLM 开始] 输入: Human: 用一句话介绍 Python...
[LLM 结束] 输出: Python是一种广泛用于数据科学、Web开发、自动化等领域的高级、通用、解释型编程语言，以其简洁易...
回复: Python是一种广泛用于数据科学、Web开发、自动化等领域的高级、通用、解释型编程语言，以其简洁易读的语法和强大的生态系统而闻名。


**术语：**
- **BaseCallbackHandler** = 回调基类，继承它并覆写需要的方法
- **on_llm_start / on_llm_end** = LLM 开始/结束时触发
- **config={"callbacks": [...]}** = 把回调传给本次调用

### 2. 耗时统计——测量每次调用的时间

In [3]:
class TimingHandler(BaseCallbackHandler):
    def on_llm_start(self, serialized, prompts, **kwargs):
        self.start = time.time()

    def on_llm_end(self, response, **kwargs):
        elapsed = time.time() - self.start
        print(f"耗时: {elapsed:.2f}s")

timmer = TimingHandler()
result = llm.invoke("请写一段 100 字左右的短文", config={"callbacks": [timmer]})
print("回复:", result.content)

耗时: 1.90s
回复: # 秋日黄昏

黄昏时分，我独自走在林间小径上。夕阳斜照，将树影拉得细长。落叶铺满路面，踩上去沙沙作响，像在诉说秋天的秘密。远处传来几声鸟鸣，清脆而悠远。风起时，几片金黄的叶子打着旋儿飘落，轻轻落在肩头。我停下脚步，深深吸了一口微凉的空气——这秋日的黄昏，安静得让人想落泪，却又温柔得让人微笑。


**术语：**
- **start / end** 配对使用，可以计算耗时、统计 token 数等
- 所有回调都支持这种开始到结束的模式

### 3. 链级别回调——监控整个链

In [5]:
from langchain_core.callbacks import BaseCallbackHandler

class ChainMonitor(BaseCallbackHandler):
    """监听 LLM 级别的回调"""
    def on_llm_start(self, serialized, prompts, **kwargs):
        print(f"[LLM 开始] 输入: {prompts[0][:50]}...")
    def on_llm_end(self, response, **kwargs):
        print(f"[LLM 结束] 输出: {response.generations[0][0].text[:50]}...")

chain = ChatPromptTemplate.from_template("解释{topic}是什么") | llm | StrOutputParser()
monitor = ChainMonitor()

# 用 with_config 传入回调，避免 bind 冲突
llm_with_config = llm.with_config(callbacks=[monitor])
chain2 = ChatPromptTemplate.from_template("解释{topic}是什么") | llm_with_config | StrOutputParser()
result = chain2.invoke({"topic": "回调函数"})
print("最终结果:", result[:80])


[LLM 开始] 输入: Human: 解释回调函数是什么...
[LLM 结束] 输出: 我们来用一个生活中的例子来解释回调函数，然后再看它在编程中的定义。

### 生活中的例子：点外卖
...
最终结果: 我们来用一个生活中的例子来解释回调函数，然后再看它在编程中的定义。

### 生活中的例子：点外卖

想象一下，你点了一份外卖。

1.  **你打电话给餐厅下


**术语：**
- **on_chain_start / on_chain_end** = 整个链（Chain）开始/结束时的回调
- 链回调可以监控整条链的执行，而 LLM 回调只监控 LLM 部分
- 回调可以挂在不同层级：LLM 级别、Chain 级别、全局级别

### 4. 全局回调——所有调用自动生效

In [9]:
class GlobalLogger(BaseCallbackHandler):
    def on_llm_start(self, serialized, prompts, **kwargs):
        # LangChain 1.0 中 serialized 格式不同, 改用 kwargs 获取模型信息
        model = kwargs.get("invocation_params", {}).get("model", "unknown")
        print(f"[全局] 调用模型: {model}")

global_logger = GlobalLogger()
llm_with_config = get_llm().with_config(callbacks=[global_logger])

r1 = llm_with_config.invoke("Hello")
print("回复1:", r1.content)
r2 = llm_with_config.invoke("Hi again")
print("回复2:", r2.content)


[全局] 调用模型: deepseek-chat
回复1: 你好！有什么可以帮你的吗？😊
[全局] 调用模型: deepseek-chat
回复2: Hello again! How can I help you today?


**术语：**
- **CallbackManager** = 回调管理器，把多个回调组合在一起
- 全局回调 = 对所有 LLM 调用自动生效，不需要每次传 config
- 实际项目中也可以把回调设置在 get_llm() 中

### 5. 实战：组合多个回调

In [10]:
class TimingHandler(BaseCallbackHandler):
    """记录耗时"""
    def on_llm_start(self, *args, **kwargs):
        self.start = time.time()
    def on_llm_end(self, *args, **kwargs):
        print(f"  ⏱ 耗时: {time.time() - self.start:.2f}s")

class TokenCountHandler(BaseCallbackHandler):
    """统计 token 数"""
    def __init__(self):
        self.count = 0
    def on_llm_new_token(self, token: str, **kwargs):
        self.count += 1
    def on_llm_end(self, *args, **kwargs):
        print(f"  📝 Token 数: {self.count}")

print("组合回调输出：")
result = llm.invoke("请用 50 字介绍量子计算", config={"callbacks": [TimingHandler(), TokenCountHandler()]})
print("回复:", result.content)

组合回调输出：
  ⏱ 耗时: 1.10s
  📝 Token 数: 0
回复: 量子计算利用量子比特的叠加与纠缠特性，可并行处理海量数据。它在药物研发、密码破译等领域潜力巨大，有望解决传统计算机难以企及的复杂问题。


**术语：**
- 多个回调可以同时使用，各自监听不同的事件
- CallbackManager 把多个回调组合成一个传给 config
- 实际项目中常用 LangSmith 做追踪（自带日志、耗时、token 统计）